# Lesson 01 - AI এজেন্টদের পরিচিতি

**AI Agents for Beginners** কোর্সের প্রথম পাঠে স্বাগতম!

একটি **AI এজেন্ট** হল একটি প্রোগ্রাম যা একটি বড় ভাষা মডেল (LLM) কে তার যুক্তি ইঞ্জিন হিসেবে ব্যবহার করে এবং বাস্তব জগতে *কার্য* নিতে পারে — API কল করা, ডেটাবেস থেকে তথ্য অনুসন্ধান করা, অথবা কোড চালানো — ব্যবহারকারীর পক্ষে একটি লক্ষ্য অর্জনের জন্য।

এই নোটবুকে আপনি আপনার প্রথম এজেন্ট তৈরি করবেন: একটি **ট্রাভেল এজেন্ট** যা ছুটির গন্তব্যস্থান সুপারিশ করবে। Along the way আপনি শিখবেন কিভাবে:

1. **Microsoft Agent Framework** ব্যবহার করে Azure AI Foundry Agent Service এর সাথে সংযোগ স্থাপন করবেন।
2. এজেন্টকে একটি **টুল** দিবেন — একটি সাধারণ পাইথন ফাংশন যেটি এটি কল করতে পারে।
3. এজেন্টকে চালাবেন এবং এর প্রতিক্রিয়া পর্যালোচনা করবেন।
4. এজেন্টের প্রতিক্রিয়া টোকেন-বাই-টোকেন স্ট্রিম করবেন।


## Setup

এই নোটবুক চালানোর আগে, নিশ্চিত করুন আপনার কাছে:

1. **একটি Azure AI Foundry প্রকল্প** যার একটি ডিপ্লয় করা চ্যাট মডেল রয়েছে (যেমন `gpt-4o-mini`)।
2. **Azure CLI-তে লগইন করেছেন** — আপনার টার্মিনালে `az login` চালান।
3. **প্রয়োজনীয় পরিবেশ ভেরিয়েবল সেট করেছেন:**
   - `AZURE_AI_PROJECT_ENDPOINT` — আপনার Azure AI Foundry প্রকল্পের এন্ডপয়েন্ট।
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — আপনার ডিপ্লয় করা মডেলের নাম।

নিচের সেলটিতে আপনার প্রয়োজনীয় পিথন প্যাকেজগুলি ইনস্টল করা হয়।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## আপনার প্রথম এজেন্ট তৈরি করা

একটি এজেন্টের দুটি জিনিস প্রয়োজন:

- **নির্দেশাবলী** যা তাকে বলে *সে কে* এবং *কিভাবে আচরণ করতে হবে* (একটি সিস্টেম প্রম্পট)।
- **টুলস** — Python ফাংশন যা `@tool` দিয়ে ডেকোরেট করা হয় এবং এজেন্ট তথ্য সংগ্রহ করতে বা কর্মসূচি সম্পাদন করতে এগুলো ব্যবহার করতে পারে।

নীচে আমরা একটি সাধারণ টুল সংজ্ঞায়িত করেছি যা জনপ্রিয় অবকাশ গন্তব্যের একটি তালিকা 반환 করে। যখন ব্যবহারকারী ভ্রমণ সুপারিশের জন্য অনুরোধ করবেন, তখন এজেন্ট এই টুলটি ব্যবহার করবে।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## স্ট্রিমিং প্রতিক্রিয়া

আরও ইন্টারেক্টিভ অভিজ্ঞতার জন্য আপনি এজেন্টের প্রতিক্রিয়া **স্ট্রিম** করতে পারেন। সম্পূর্ণ উত্তরটির জন্য অপেক্ষা করার পরিবর্তে, এজেন্ট তৈরি হওয়া টেক্সট অংশগুলি ধাপে ধাপে প্রদান করে। এটি বিশেষভাবে চ্যাট ইন্টারফেসে উপকারী যেখানে আপনি ফলাফল রিয়েল টাইমে প্রদর্শন করতে চান।


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## সারাংশ

এই পাঠে আপনি শিখেছেন কিভাবে:

- **একটি প্রোভাইডার তৈরি করবেন** যা `AzureAIProjectAgentProvider` এর মাধ্যমে Azure AI Foundry Agent Service এর সাথে সংযুক্ত হয়।
- **একটি টুল সংজ্ঞায়িত করবেন** `@tool` ডেকোরেটরের সাহায্যে যাতে এজেন্ট আপনার পাইথন ফাংশনগুলো কল করতে পারে।
- **এজেন্ট চলাবেন** একটি ব্যবহারকারী বার্তার সাথে এবং এর প্রতিক্রিয়া প্রিন্ট করবেন।
- **রিয়েল-টাইম আউটপুটের জন্য রেসপন্স স্ট্রিমিং করবেন**।

পরবর্তী পাঠে আমরা এজেন্টিক ফ্রেমওয়ার্কগুলি আরও গভীরভাবে অন্বেষণ করব এবং শিখব কিভাবে এজেন্টদের আরও শক্তিশালী টুল এবং বহু-ধাপ যুক্তি সম্পন্ন ক্ষমতা দেওয়া যায়।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
